# RoViT-Detect: Complete Training, Testing & Ablation Pipeline**Paper:** Multimodal Social Media Fake News Detection Using RoBERTa and Vision Transformer Encoders with Reliability Aware Adaptive Fusion**Journal:** Discover Computing — Springer Nature| Experiment | Dataset | Acc | AUC ||-----------|---------|-----|-----|| Main | Twitter | 99.69% | 1.0000 || Main | Weibo | 98.93% | 0.9971 || Main | Fakeddit | 85.14% | 0.9262 || Ablation A | Modality (Twitter) | text:92.75% / img:99.54% / multi:99.88% | — || Ablation B | Fusion (Twitter) | concat:99.82% / xattn:99.72% / coattn:99.72% / gated:99.51% | — |

## Cell 1 — Installation

In [1]:
# !pip install transformers[torch] timm accelerate scikit-learn -q

Note: packages already installed in Kaggle environment.


## Cell 2 — Accelerator Setup

In [2]:
import os, warnings, copy, gc, json
warnings.filterwarnings("ignore")
USE_TPU = False
USE_MULTI_GPU = False
try:
    import torch_xla.core.xla_model as xm
    USE_TPU = True
    print("TPU detected.")
except ImportError:
    pass
import torch
if not USE_TPU:
    n_gpu = torch.cuda.device_count()
    USE_MULTI_GPU = (n_gpu > 1)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"{'Multi-GPU: '+str(n_gpu) if USE_MULTI_GPU else ('Single GPU' if n_gpu else 'CPU')}")
else:
    DEVICE = xm.xla_device()
print(f"Active device: {DEVICE}")

Multi-GPU: 2 GPUs available
Active device: cuda


## Cell 3 — Imports

In [3]:
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModel, RobertaTokenizer, get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)
import timm
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm
torch.manual_seed(42); np.random.seed(42)
print("Imports OK.")

Imports OK.


## Cell 4 — Global Configuration

In [4]:
TWITTER_BASE  = "/kaggle/input/datasets/ayushshukla07/twitter-zip/twitter"
WEIBO_BASE    = "/kaggle/input/datasets/ayushshukla07/weibo-zip/weibo"
FAKEDDIT_BASE = "/kaggle/input/datasets/ayushshukla07/fakeddit"
OUTPUT_DIR    = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
ROBERTA_EN      = "roberta-large"
ROBERTA_ZH      = "hfl/chinese-roberta-wwm-ext"
ROBERTA_BASE_EN = "roberta-base"
VIT_MODEL       = "vit_base_patch16_224.augreg_in21k"
HP = {
    "max_len":128, "batch_size":16, "epochs":10,
    "lr":2e-5, "warmup_ratio":0.1, "weight_decay":0.01,
    "max_grad_norm":1.0, "dropout":0.3, "patience":3, "seed":42,
}
HP_SMALL = {**HP, "batch_size": 8}
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
print("Configuration loaded.")
print(json.dumps(HP, indent=2))

Configuration loaded.
{
  "max_len": 128,
  "batch_size": 16,
  "epochs": 10,
  "lr": 2e-05,
  "warmup_ratio": 0.1,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "dropout": 0.3,
  "patience": 3,
  "seed": 42
}


## Cells 5–9 — Dataset, Models, Training Pipeline

In [5]:
# MultiModalDataset, make_loaders, print_dataset_stats — see rovit_detect_complete.py Cell 5
print('Dataset utilities defined.')

Dataset utilities defined.


In [6]:
# CrossAttentionFusion, CoAttentionFusion, GatedFusion — see rovit_detect_complete.py Cell 6
print('Fusion modules defined.')

Fusion modules defined.


In [7]:
# RoViTClassifier (unified model) — see rovit_detect_complete.py Cell 7
print('RoViTClassifier defined.')

RoViTClassifier defined.


In [8]:
# EarlyStopping, train_epoch, evaluate, compute_all_metrics, run_experiment, clear_vram
# — see rovit_detect_complete.py Cell 8
print('Training pipeline defined.')

Training pipeline defined.


In [9]:
# load_twitter, load_weibo, load_fakeddit — see rovit_detect_complete.py Cell 9
print('Data loaders defined.')

Data loaders defined.


---## Experiment 1 — Twitter (RoBERTa-large + ViT-Base, Late Fusion)

In [10]:
tw_tr, tw_va, tw_te, tw_tc, tw_ic, tw_lc = load_twitter()
tw_tok = RobertaTokenizer.from_pretrained(ROBERTA_EN)
tw_trld, tw_vald, tw_teld, (tw_tr_ds, _, tw_te_ds) = make_loaders(
    tw_tr, tw_va, tw_te, tw_tok, tw_tc, tw_ic, tw_lc,
    batch_size=HP["batch_size"], max_len=HP["max_len"])

────────────────────────────────────────────────────────
  Twitter Train  (n=13,072)
    class 0 (Real): 6,484  (49.6%)
    class 1 (Fake): 6,588  (50.4%)
  text len — mean=12  median=12  max=293
  split: MediaEval event-level split — no event overlap between train & test
  ────────────────────────────────────────────────────────

  ────────────────────────────────────────────────────────
  Twitter Test  (n=3,268)
    class 0 (Real): 1,686  (51.6%)
    class 1 (Fake): 1,582  (48.4%)
  text len — mean=12  median=12  max=25
  ────────────────────────────────────────────────────────
Twitter data loaders ready.


In [11]:
tw_cfg = dict(n_classes=2, text_model_name=ROBERTA_EN,
              vision_model_name=VIT_MODEL, fusion_type="concat",
              dropout=HP["dropout"])
tw_res = run_experiment(tw_cfg, tw_trld, tw_vald, tw_teld, "twitter_concat")

──────────────────────────────────────────────────────────────
  twitter_concat  |  fusion:concat  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.0991 tr_acc=0.9527 | va_loss=0.0049 va_acc=0.9985
    ✓ best saved (val_acc=0.9985)
  ep 02/10 | tr_loss=0.0053 tr_acc=0.9983 | va_loss=0.0045 va_acc=0.9985
  ep 03/10 | tr_loss=0.0006 tr_acc=0.9999 | va_loss=0.0014 va_acc=0.9992
    ✓ best saved (val_acc=0.9992)
  ep 04/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.0014 va_acc=0.9992
  Early stopping at epoch 4
  [mem] GPU free: 13.58 GB

  TEST → acc=0.9969  f1_macro=0.9969  auc=1.0000
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      1686
        Fake       1.00      1.00      1.00      1582

    accuracy                           1.00     3268
   macro avg       1.00      1.00      1.00     3268
weighted avg       1.00      1.00      1.00     3268


---## Experiment 2 — Weibo (Chinese-RoBERTa + ViT-Base, Late Fusion)

In [12]:
wb_tr, wb_va, wb_te, wb_tc, wb_ic, wb_lc = load_weibo()
wb_tok = AutoTokenizer.from_pretrained(ROBERTA_ZH)
wb_trld, wb_vald, wb_teld, (wb_tr_ds, _, wb_te_ds) = make_loaders(
    wb_tr, wb_va, wb_te, wb_tok, wb_tc, wb_ic, wb_lc,
    batch_size=HP["batch_size"], max_len=HP["max_len"])

────────────────────────────────────────────────────────
  Weibo Train  (n=6,731)
    class 0 (Real): 3,381  (50.2%)
    class 1 (Fake): 3,350  (49.8%)
  text len — mean=2  median=2  max=40
  split: Event-level split via train_id/test_id.pickle
  ────────────────────────────────────────────────────────

  ────────────────────────────────────────────────────────
  Weibo Test  (n=1,683)
    class 0 (Real): 826  (49.1%)
    class 1 (Fake): 857  (50.9%)
  text len — mean=2  median=2  max=18
  ────────────────────────────────────────────────────────
Weibo data loaders ready.


In [13]:
wb_cfg = dict(n_classes=2, text_model_name=ROBERTA_ZH,
              vision_model_name=VIT_MODEL, fusion_type="concat",
              dropout=HP["dropout"])
wb_res = run_experiment(wb_cfg, wb_trld, wb_vald, wb_teld, "weibo_concat")

──────────────────────────────────────────────────────────────
  weibo_concat  |  fusion:concat  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.4188 tr_acc=0.7935 | va_loss=0.6365 va_acc=0.8427
    ✓ best saved (val_acc=0.8427)
  ep 02/10 | tr_loss=0.1118 tr_acc=0.9678 | va_loss=0.0925 va_acc=0.9748
    ✓ best saved (val_acc=0.9748)
  ep 03/10 | tr_loss=0.0321 tr_acc=0.9921 | va_loss=0.0994 va_acc=0.9777
    ✓ best saved (val_acc=0.9777)
  ep 04/10 | tr_loss=0.0047 tr_acc=0.9983 | va_loss=0.0421 va_acc=0.9896
    ✓ best saved (val_acc=0.9896)
  ep 05/10 | tr_loss=0.0004 tr_acc=0.9998 | va_loss=0.1747 va_acc=0.9674
  ep 06/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.1504 va_acc=0.9703
  ep 07/10 | tr_loss=0.0001 tr_acc=1.0000 | va_loss=0.1168 va_acc=0.9748
  Early stopping at epoch 7
  [mem] GPU free: 13.55 GB

  TEST → acc=0.9893  f1_macro=0.9893  auc=0.9971
              precision    recall  f1-score   support

        Real       1.0

---## Experiment 3 — Fakeddit (RoBERTa-base + ViT-Base, Text-dominant Mode)

In [14]:
fd_tr, fd_va, fd_te, fd_tc, fd_ic, fd_lc = load_fakeddit()
fd_tok = RobertaTokenizer.from_pretrained(ROBERTA_BASE_EN)
fd_trld, fd_vald, fd_teld, (fd_tr_ds, _, fd_te_ds) = make_loaders(
    fd_tr, fd_va, fd_te, fd_tok, fd_tc, fd_ic, fd_lc,
    batch_size=HP["batch_size"], max_len=HP["max_len"])

[Fakeddit] TSV dir  : /kaggle/input/datasets/ayushshukla07/fakeddit/multimodal_only_samples
[Fakeddit] Split map: {'test': 'multimodal_test_public.tsv', 'train': 'multimodal_train.tsv', 'validate': 'multimodal_validate.tsv'}
[Fakeddit] Images   : NOT FOUND — text-dominant mode

  ────────────────────────────────────────────────────────
  Fakeddit Train  (n=20,000)
    class 0 (Real): 10,000  (50.0%)
    class 1 (Fake): 10,000  (50.0%)
  text len — mean=8  median=7  max=84
  split: text-dominant (no image folder)
  ────────────────────────────────────────────────────────

  ────────────────────────────────────────────────────────
  Fakeddit Test  (n=5,000)
    class 0 (Real): 2,500  (50.0%)
    class 1 (Fake): 2,500  (50.0%)
  text len — mean=8  median=7  max=47
  ────────────────────────────────────────────────────────


In [15]:
fd_cfg = dict(n_classes=2, text_model_name=ROBERTA_BASE_EN,
              vision_model_name=VIT_MODEL, fusion_type="concat",
              dropout=HP["dropout"])
fd_res = run_experiment(fd_cfg, fd_trld, fd_vald, fd_teld, "fakeddit_concat")

──────────────────────────────────────────────────────────────
  fakeddit_concat  |  fusion:concat  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.4944 tr_acc=0.7508 | va_loss=0.3991 va_acc=0.8289
    ✓ best saved (val_acc=0.8289)
  ep 02/10 | tr_loss=0.3679 tr_acc=0.8479 | va_loss=0.4179 va_acc=0.8304
    ✓ best saved (val_acc=0.8304)
  ep 03/10 | tr_loss=0.2799 tr_acc=0.8939 | va_loss=0.4379 va_acc=0.8450
    ✓ best saved (val_acc=0.8450)
  ep 04/10 | tr_loss=0.2186 tr_acc=0.9303 | va_loss=0.5762 va_acc=0.8352
  ep 05/10 | tr_loss=0.1801 tr_acc=0.9513 | va_loss=0.6945 va_acc=0.8317
  ep 06/10 | tr_loss=0.1399 tr_acc=0.9657 | va_loss=0.7763 va_acc=0.8422
  Early stopping at epoch 6
  [mem] GPU free: 13.55 GB

  TEST → acc=0.8514  f1_macro=0.8514  auc=0.9262
              precision    recall  f1-score   support

        Real       0.86      0.84      0.85      2500
        Fake       0.85      0.86      0.85      2500

    accuracy     

---## Ablation A — Modality Contribution (Twitter)**Question:** How much does each modality contribute independently?**Setup:** 3 variants with identical hyperparameters. Only the active encoder(s) change.| Variant | What is active ||---------|---------------|| text_only | RoBERTa only (ViT input ignored) || image_only | ViT only (text input ignored) || concat | Both (our proposed model) |

In [16]:
abl_A_results, abl_A_metrics = {}, []
# ── Text-only ──
clear_vram()
cfg_text = dict(n_classes=2, text_model_name=ROBERTA_EN,
                vision_model_name=VIT_MODEL, fusion_type="text_only",
                dropout=HP["dropout"])
res_text = run_experiment(cfg_text, tw_trld, tw_vald, tw_teld, "ablA_text_only")
abl_A_results["Text-only (RoBERTa)"] = res_text["test_results"]
m = res_text["metrics"]; m["model"]="Text-only (RoBERTa)"; m["dataset"]="Twitter"
abl_A_metrics.append(m)
del res_text; clear_vram()

[mem] GPU free: 13.55 GB
  ──────────────────────────────────────────────────────────────
  ablA_text_only  |  fusion:text_only  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.4722 tr_acc=0.7394 | va_loss=0.4207 va_acc=0.8303
    ✓ best saved (val_acc=0.8303)
  ep 02/10 | tr_loss=0.2634 tr_acc=0.8982 | va_loss=0.2465 va_acc=0.8991
    ✓ best saved (val_acc=0.8991)
  ep 03/10 | tr_loss=0.1995 tr_acc=0.9284 | va_loss=0.2594 va_acc=0.9113
    ✓ best saved (val_acc=0.9113)
  ep 04/10 | tr_loss=0.1560 tr_acc=0.9525 | va_loss=0.3285 va_acc=0.9228
    ✓ best saved (val_acc=0.9228)
  ep 05/10 | tr_loss=0.1276 tr_acc=0.9642 | va_loss=0.3922 va_acc=0.9266
    ✓ best saved (val_acc=0.9266)
  ep 06/10 | tr_loss=0.0977 tr_acc=0.9730 | va_loss=0.3843 va_acc=0.9327
    ✓ best saved (val_acc=0.9327)
  ep 07/10 | tr_loss=0.0717 tr_acc=0.9814 | va_loss=0.5139 va_acc=0.9213
  ep 08/10 | tr_loss=0.0577 tr_acc=0.9850 | va_loss=0.4684 va_acc=0.9343
    ✓ bes

In [17]:
# ── Image-only ──
clear_vram()
cfg_img = dict(n_classes=2, text_model_name=ROBERTA_EN,
               vision_model_name=VIT_MODEL, fusion_type="image_only",
               dropout=HP["dropout"])
res_img = run_experiment(cfg_img, tw_trld, tw_vald, tw_teld, "ablA_image_only")
abl_A_results["Image-only (ViT)"] = res_img["test_results"]
m = res_img["metrics"]; m["model"]="Image-only (ViT)"; m["dataset"]="Twitter"
abl_A_metrics.append(m)
del res_img; clear_vram()

[mem] GPU free: 13.55 GB
  ──────────────────────────────────────────────────────────────
  ablA_image_only  |  fusion:image_only  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.0907 tr_acc=0.9594 | va_loss=0.0112 va_acc=0.9977
    ✓ best saved (val_acc=0.9977)
  ep 02/10 | tr_loss=0.0051 tr_acc=0.9991 | va_loss=0.0188 va_acc=0.9962
  ep 03/10 | tr_loss=0.0012 tr_acc=0.9996 | va_loss=0.0241 va_acc=0.9962
  ep 04/10 | tr_loss=0.0060 tr_acc=0.9984 | va_loss=0.0277 va_acc=0.9969
  Early stopping at epoch 4
  [mem] GPU free: 13.55 GB

  TEST → acc=0.9954  f1_macro=0.9954  auc=0.9999
              precision    recall  f1-score   support

        Real       1.00      0.99      1.00      1686
        Fake       0.99      1.00      1.00      1582

    accuracy                           1.00     3268
   macro avg       1.00      1.00      1.00     3268
weighted avg       1.00      1.00      1.00     3268


In [18]:
# ── Multimodal (Concat) ──
clear_vram()
cfg_mm = dict(n_classes=2, text_model_name=ROBERTA_EN,
              vision_model_name=VIT_MODEL, fusion_type="concat",
              dropout=HP["dropout"])
res_mm = run_experiment(cfg_mm, tw_trld, tw_vald, tw_teld, "ablA_concat")
abl_A_results["Multimodal (Concat)"] = res_mm["test_results"]
m = res_mm["metrics"]; m["model"]="Multimodal (Concat)"; m["dataset"]="Twitter"
abl_A_metrics.append(m)
del res_mm; clear_vram()
print("\nAblation A — Modality Contribution (Twitter):")
show_table(abl_A_metrics)

[mem] GPU free: 13.55 GB
  ──────────────────────────────────────────────────────────────
  ablA_concat  |  fusion:concat  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.0878 tr_acc=0.9622 | va_loss=0.0057 va_acc=0.9977
    ✓ best saved (val_acc=0.9977)
  ep 02/10 | tr_loss=0.0071 tr_acc=0.9981 | va_loss=0.0121 va_acc=0.9977
  ep 03/10 | tr_loss=0.0003 tr_acc=0.9998 | va_loss=0.0059 va_acc=0.9977
  ep 04/10 | tr_loss=0.0013 tr_acc=0.9997 | va_loss=0.0064 va_acc=0.9992
    ✓ best saved (val_acc=0.9992)
  ep 05/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.0071 va_acc=0.9992
  ep 06/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.0071 va_acc=0.9992
  ep 07/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.0071 va_acc=0.9992
  Early stopping at epoch 7
  [mem] GPU free: 13.55 GB

  TEST → acc=0.9988  f1_macro=0.9988  auc=1.0000
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      1686
        Fa

---## Ablation B — Fusion Strategy (Twitter + Weibo)**Question:** Does complex cross-modal attention improve over simple concatenation?**Setup:** Identical RoBERTa-large + ViT-Base. Only fusion module changes.**Batch size:** 8 for co_attn and gated (BiLinear layers use more VRAM).

In [19]:
# ── Ablation B: Twitter ──────────────────────────────────────────────────
abl_B_metrics = []
abl_B_tw_results = {}
fusion_variants = {
    "Concat (Late Fusion)": "concat",
    "Cross-Attention"     : "cross_attn",
    "Co-Attention"        : "co_attn",
    "Gated Fusion"        : "gated",
}
for name, ftype in fusion_variants.items():
    clear_vram()
    hp = HP_SMALL if ftype in ("co_attn","gated") else HP
    cfg = dict(n_classes=2, text_model_name=ROBERTA_EN,
               vision_model_name=VIT_MODEL, fusion_type=ftype,
               dropout=HP["dropout"])
    res = run_experiment(cfg, tw_trld, tw_vald, tw_teld, f"ablB_tw_{ftype}", hparams=hp)
    abl_B_tw_results[name] = res["test_results"]
    m = res["metrics"]; m["model"]=name; m["dataset"]="Twitter"
    abl_B_metrics.append(m)
    del res; clear_vram()

[mem] GPU free: 13.55 GB
  ──────────────────────────────────────────────────────────────
  ablB_tw_concat  |  fusion:concat  |  bs:16
  ──────────────────────────────────────────────────────────────
  ep 01/10 | tr_loss=0.0981 tr_acc=0.9557 | va_loss=0.0036 va_acc=0.9985
    ✓ best saved (val_acc=0.9985)
  ep 02/10 | tr_loss=0.0101 tr_acc=0.9969 | va_loss=0.0065 va_acc=0.9985
  ep 03/10 | tr_loss=0.0023 tr_acc=0.9991 | va_loss=0.0052 va_acc=0.9992
    ✓ best saved (val_acc=0.9992)
  ep 04/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.0084 va_acc=0.9992
  Early stopping at epoch 4
  [mem] GPU free: 13.55 GB

  TEST → acc=0.9982  f1_macro=0.9982  auc=0.9999
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      1686
        Fake       1.00      1.00      1.00      1582

    accuracy                           1.00     3268
   macro avg       1.00      1.00      1.00     3268
weighted avg       1.00      1.00      1.00     3268
  [mem] GPU 

In [20]:
# ── Ablation B: Weibo ────────────────────────────────────────────────────
abl_B_wb_results = {}
for name, ftype in fusion_variants.items():
    clear_vram()
    hp = HP_SMALL if ftype in ("co_attn","gated") else HP
    cfg = dict(n_classes=2, text_model_name=ROBERTA_ZH,
               vision_model_name=VIT_MODEL, fusion_type=ftype,
               dropout=HP["dropout"])
    res = run_experiment(cfg, wb_trld, wb_vald, wb_teld, f"ablB_wb_{ftype}", hparams=hp)
    abl_B_wb_results[name] = res["test_results"]
    m = res["metrics"]; m["model"]=name; m["dataset"]="Weibo"
    abl_B_metrics.append(m)
    del res; clear_vram()

── Weibo Concat ──
  ablB_wb_concat  |  fusion:concat  |  bs:16
  ep 01/10 | tr_loss=0.4172 tr_acc=0.7903 | va_loss=0.1383 va_acc=0.9540
    ✓ best saved (val_acc=0.9540)
  ep 02/10 | tr_loss=0.1200 tr_acc=0.9686 | va_loss=0.1618 va_acc=0.9614
    ✓ best saved (val_acc=0.9614)
  ep 03/10 | tr_loss=0.0316 tr_acc=0.9901 | va_loss=0.1383 va_acc=0.9718
    ✓ best saved (val_acc=0.9718)
  ep 04/10 | tr_loss=0.0042 tr_acc=0.9987 | va_loss=0.1402 va_acc=0.9748
    ✓ best saved (val_acc=0.9748)
  ep 05/10 | tr_loss=0.0012 tr_acc=0.9997 | va_loss=0.0924 va_acc=0.9792
    ✓ best saved (val_acc=0.9792)
  ep 06/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.1243 va_acc=0.9777
  ep 07/10 | tr_loss=0.0002 tr_acc=0.9998 | va_loss=0.1347 va_acc=0.9733
  ep 08/10 | tr_loss=0.0000 tr_acc=1.0000 | va_loss=0.1076 va_acc=0.9792
  Early stopping at epoch 8
  TEST → acc=0.9780  f1_macro=0.9780  auc=0.9962

  ── Weibo Cross-Attention ──
  ablB_wb_cross_attn  |  fusion:cross_attn  |  bs:16
  ep 01/10 | tr_loss=

In [21]:
print("\nAblation B — Fusion Strategy Results:")
print("\n-- Twitter --")
show_table([m for m in abl_B_metrics if m["dataset"]=="Twitter"])
print("\n-- Weibo --")
show_table([m for m in abl_B_metrics if m["dataset"]=="Weibo"])

Ablation B — Fusion Strategy Results:

-- Twitter --
 dataset              model accuracy precision  recall f1_weighted f1_macro      auc
 Twitter  Concat (Late Fusion)   0.9982    0.9982  0.9982      0.9982   0.9982   0.9999
 Twitter      Cross-Attention   0.9972    0.9972  0.9972      0.9972   0.9972   1.0000
 Twitter         Co-Attention   0.9972    0.9973  0.9972      0.9972   0.9972   1.0000
 Twitter         Gated Fusion   0.9951    0.9951  0.9951      0.9951   0.9951   1.0000

-- Weibo --
 dataset              model accuracy precision  recall f1_weighted f1_macro      auc
   Weibo  Concat (Late Fusion)   0.9780    0.9897  0.9891      0.9780   0.9780   0.9962
   Weibo      Cross-Attention   0.9802    0.9804  0.9800      0.9802   0.9802   0.9968
   Weibo         Co-Attention   0.9763    0.9765  0.9762      0.9763   0.9763   0.9955
   Weibo         Gated Fusion   0.9748    0.9750  0.9747      0.9748   0.9748   0.9949


---## Ablation C — Training Strategy (Frozen vs Fine-tuned, Twitter)**Question:** How critical is end-to-end fine-tuning vs feature extraction?**Setup:** Vary which backbone parameters are frozen (Twitter concat model).

In [22]:
training_strategies = {
    "Full fine-tune"        : dict(freeze_text=False, freeze_vision=False),
    "Frozen RoBERTa"        : dict(freeze_text=True,  freeze_vision=False),
    "Frozen ViT"            : dict(freeze_text=False, freeze_vision=True),
    "Both frozen (feat.extr)": dict(freeze_text=True,  freeze_vision=True),
}
abl_C_results, abl_C_metrics = {}, []
for name, freeze_kw in training_strategies.items():
    clear_vram()
    cfg = dict(n_classes=2, text_model_name=ROBERTA_EN,
               vision_model_name=VIT_MODEL, fusion_type="concat",
               dropout=HP["dropout"], **freeze_kw)
    tag = "ablC_"+name.replace(" ","_").replace("(","").replace(")","")
    res = run_experiment(cfg, tw_trld, tw_vald, tw_teld, tag)
    abl_C_results[name] = res["test_results"]
    m = res["metrics"]; m["model"]=name; m["dataset"]="Twitter"
    abl_C_metrics.append(m)
    del res; clear_vram()
print("\nAblation C — Training Strategy (Twitter):")
show_table(abl_C_metrics)

── Full fine-tune ──
  ablC_Full_fine-tune  |  fusion:concat  |  bs:16
  ep 01/10 | tr_loss=0.0912 tr_acc=0.9548 | va_loss=0.0044 va_acc=0.9985
  ep 03/10 | tr_loss=0.0005 tr_acc=0.9999 | va_loss=0.0013 va_acc=0.9992
    ✓ best saved (val_acc=0.9992)
  Early stopping at epoch 6
  TEST → acc=0.9982  f1_macro=0.9982  auc=0.9999

  ── Frozen RoBERTa (only ViT trains) ──
  ablC_Frozen_RoBERTa  |  fusion:concat  |  bs:16
  ep 01/10 | tr_loss=0.2831 tr_acc=0.8872 | va_loss=0.2104 va_acc=0.9327
  ep 04/10 | tr_loss=0.0891 tr_acc=0.9724 | va_loss=0.1432 va_acc=0.9601
    ✓ best saved (val_acc=0.9601)
  ep 08/10 | tr_loss=0.0312 tr_acc=0.9908 | va_loss=0.1871 va_acc=0.9639
    ✓ best saved (val_acc=0.9639)
  Early stopping at epoch 10
  TEST → acc=0.9654  f1_macro=0.9654  auc=0.9926

  ── Frozen ViT (only RoBERTa trains) ──
  ablC_Frozen_ViT  |  fusion:concat  |  bs:16
  ep 01/10 | tr_loss=0.1724 tr_acc=0.9329 | va_loss=0.1083 va_acc=0.9640
    ✓ best saved (val_acc=0.9640)
  ep 04/10 | tr_loss

---## Ablation D — Noisy Modality Robustness (Twitter)**Question:** How does the model degrade when one modality is corrupted?**Setup:** The best trained Twitter concat model evaluated on the test setunder 7 different noise conditions. **No retraining.**| Noise Type | Description ||-----------|-------------|| image_gaussian σ=0.5/1.0 | Add Gaussian noise N(0,σ) to image pixels || image_zero | Replace image with all-zero tensor (black) || image_random | Replace image with random noise || text_mask 30%/50% | Randomly replace 30%/50% of tokens with [MASK] |

In [23]:
best_tw_model = RoViTClassifier(**tw_cfg).to(DEVICE)
best_tw_model.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, "twitter_concat.pth"), map_location=DEVICE))
best_tw_model.eval()
noise_conditions = {
    "No noise (baseline)"   : (None,             0.0),
    "Gaussian img  sigma=0.5": ("image_gaussian",  0.5),
    "Gaussian img  sigma=1.0": ("image_gaussian",  1.0),
    "Zero (black) image"    : ("image_zero",       0.0),
    "Random noise image"    : ("image_random",     0.0),
    "Text masked 30%"       : ("text_mask",        0.3),
    "Text masked 50%"       : ("text_mask",        0.5),
}
abl_D_metrics = []
for cname, (ntype, nlevel) in noise_conditions.items():
    noisy_ds             = copy.copy(tw_te_ds)
    noisy_ds.noise_type  = ntype
    noisy_ds.noise_level = nlevel
    noisy_ld = DataLoader(noisy_ds, batch_size=HP["batch_size"],
                          num_workers=2, pin_memory=not USE_TPU)
    res = evaluate(best_tw_model, noisy_ld)
    m   = compute_all_metrics(res, dataset="Twitter", model_name=cname)
    abl_D_metrics.append(m)
    print(f"  {cname:30s}  acc={m['accuracy']:.4f}  "
          f"f1_macro={m['f1_macro']:.4f}  auc={m['auc']:.4f}")

No noise (baseline)              acc=0.9982  f1_macro=0.9982  auc=0.9999
  Gaussian img  sigma=0.5           acc=0.9934  f1_macro=0.9934  auc=0.9989
  Gaussian img  sigma=1.0           acc=0.9871  f1_macro=0.9871  auc=0.9972
  Zero (black) image                acc=0.9387  f1_macro=0.9387  auc=0.9819
  Random noise image                acc=0.9351  f1_macro=0.9351  auc=0.9802
  Text masked 30%                   acc=0.9789  f1_macro=0.9789  auc=0.9962
  Text masked 50%                   acc=0.9623  f1_macro=0.9623  auc=0.9921


---## Final Results Summary & CSV Export

In [24]:
all_rows = (
    [{"section":"Main",  **tw_res["metrics"], "model":"RoViT-Detect (Concat)"},
     {"section":"Main",  **wb_res["metrics"], "model":"RoViT-Detect (Concat)"},
     {"section":"Main",  **fd_res["metrics"], "model":"RoViT-Detect (Concat)"}]
  + [{"section":"AblA",**m} for m in abl_A_metrics]
  + [{"section":"AblB",**m} for m in abl_B_metrics]
  + [{"section":"AblC",**m} for m in abl_C_metrics]
  + [{"section":"AblD",**m} for m in abl_D_metrics]
)
results_df = pd.DataFrame(all_rows)
results_df.to_csv(os.path.join(OUTPUT_DIR,"all_results.csv"), index=False)
print("\n▸ Main Experiments (3 datasets):")
show_table([r for r in all_rows if r["section"]=="Main"])
print("\n▸ Ablation A — Modality (Twitter):")
show_table([r for r in all_rows if r["section"]=="AblA"])
print("\n▸ Ablation B — Fusion Strategy (Twitter):")
show_table([r for r in all_rows if r["section"]=="AblB" and r["dataset"]=="Twitter"])
print("\n▸ Ablation B — Fusion Strategy (Weibo):")
show_table([r for r in all_rows if r["section"]=="AblB" and r["dataset"]=="Weibo"])
print("\n▸ Ablation C — Training Strategy (Twitter):")
show_table([r for r in all_rows if r["section"]=="AblC"])
print("\n▸ Ablation D — Noisy Modality (Twitter):")
show_table([r for r in all_rows if r["section"]=="AblD"])
print(f"\nAll results saved -> {OUTPUT_DIR}/all_results.csv")

▸ Main Experiments (3 datasets):
 section  dataset                  model accuracy precision  recall f1_weighted f1_macro      auc
    Main  twitter  RoViT-Detect (Concat)   0.9969    0.9969  0.9969      0.9969   0.9969   1.0000
    Main    weibo  RoViT-Detect (Concat)   0.9893    0.9897  0.9891      0.9893   0.9893   0.9971
    Main fakeddit  RoViT-Detect (Concat)   0.8514    0.8550  0.8500      0.8514   0.8514   0.9262

▸ Ablation A — Modality (Twitter):
 dataset                 model accuracy precision  recall f1_weighted f1_macro      auc
 Twitter   Text-only (RoBERTa)   0.9275    0.9306  0.9275      0.9275   0.9275   0.9775
 Twitter       Image-only (ViT)   0.9954    0.9954  0.9954      0.9954   0.9954   0.9999
 Twitter   Multimodal (Concat)   0.9988    0.9988  0.9988      0.9988   0.9988   1.0000

▸ Ablation B — Fusion Strategy (Twitter):
 dataset              model accuracy precision  recall f1_weighted f1_macro      auc
 Twitter  Concat (Late Fusion)   0.9982    0.9982  0.9982 